# CRIVO — Exportar a voz do 17M para NumPy (site público)

O site do CRIVO roda sem PyTorch. Este notebook converte os pesos aprovados do realizador (fase 3, SHA `745a2500…`) para o formato NumPy que o site já usa em outros modelos do CRIVO (float16), confere se o NumPy dá o mesmo resultado que o PyTorch e divide o arquivo em partes de até 3,5 MB no Drive, para entrar no repositório.

**Não precisa de GPU.** Ambiente de execução → Executar tudo (uns 2 minutos). Resultado em `MyDrive/CRIVO/realizador-numpy/`.


In [ ]:
# Célula 0 — Drive e pesos aprovados
import os, sys, json, shutil, subprocess, hashlib
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
MEU = Path("/content/drive/MyDrive")
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
CRIVO = next((p for p in candidatas if (p / "transformer-17m-fase3").is_dir()), MEU / "CRIVO")
APROVADO = "745a2500615559d822f4fdb4d3d1c72850f64a9eac32a210bd6e5881f1d0b293"
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()
ORIGEM = next((p.parent for p in sorted((CRIVO / "transformer-17m-fase3").glob("fase3_*/pesos.pt")) if sha(p) == APROVADO), None)
if ORIGEM is None:
    raise SystemExit("ERRO: não achei os pesos aprovados (sha 745a2500…) em CRIVO/transformer-17m-fase3/")
LOCAL = Path("/content/realizador_pt")
shutil.rmtree(LOCAL, ignore_errors=True)
shutil.copytree(ORIGEM, LOCAL)
print("pesos aprovados:", ORIGEM, flush=True)


In [ ]:
# Célula 1 — código do CRIVO e exportação com conferência NumPy × PyTorch
REPO = Path("/content/CRIVO")
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1"], check=True)
subprocess.run([sys.executable, "scripts/exportar_pontuador.py", "--origem", str(LOCAL), "--destino", str(LOCAL)],
               cwd=REPO, check=True)
NPZ = LOCAL / "pesos_numpy.npz"
print("npz:", round(NPZ.stat().st_size / 1e6, 1), "MB | sha256", sha(NPZ), flush=True)


In [ ]:
# Célula 2 — partes de até 3,5 MB no Drive (o conector do Drive não entrega arquivos maiores)
SAIDA = CRIVO / "realizador-numpy"
SAIDA.mkdir(parents=True, exist_ok=True)
for antigo in SAIDA.glob("parte_*"):
    antigo.unlink()
dados = NPZ.read_bytes()
TAM = 3584 * 1024
partes = []
for i in range(0, len(dados), TAM):
    nome = "parte_%02d.bin" % (i // TAM)
    (SAIDA / nome).write_bytes(dados[i:i + TAM])
    partes.append({"nome": nome, "sha256": hashlib.sha256(dados[i:i + TAM]).hexdigest()})
manifesto = {"pesos_origem_sha256": APROVADO, "npz_sha256": sha(NPZ), "npz_bytes": len(dados), "partes": partes,
             "tokenizer_sha256": sha(LOCAL / "tokenizer.json")}
(SAIDA / "manifesto.json").write_text(json.dumps(manifesto, indent=1))
shutil.copyfile(LOCAL / "tokenizer.json", SAIDA / "tokenizer.json")
print(json.dumps(manifesto, indent=1))
print("\nPRONTO: me avise que eu pego as partes no Drive.", flush=True)
